# Environment Setup

In this step, all required libraries for model training and evaluation are imported. The notebook is configured to use GPU acceleration (CUDA) whenever available to significantly reduce training time.

The required packages include:

- Transformers (Hugging Face)
- Datasets
- Accelerate
- PyTorch
- Pandas
- NumPy
- Scikit-learn

This establishes the development environment for fine-tuning the MentalRoBERTa backbone.

In [7]:
!pip install -q transformers datasets accelerate scikit-learn

import torch, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


# Dataset Preparation – Dreaddit

The Dreaddit dataset is loaded for the first stage of Phase 1.

The dataset is divided into:
- Training set
- Validation set
- Test set

Only the required columns (`text` and `label`) are retained, while missing entries are removed.

A validation split is created from the original training data to monitor the model during fine-tuning and prevent overfitting.

In [8]:
train_full = pd.read_csv("/kaggle/input/datasets/ruchi798/stress-analysis-in-social-media/dreaddit-train.csv")[["text","label"]].dropna()
test_df = pd.read_csv("/kaggle/input/datasets/ruchi798/stress-analysis-in-social-media/dreaddit-test.csv")[["text","label"]].dropna()

train_df, val_df = train_test_split(train_full, test_size=0.15, stratify=train_full["label"], random_state=42)
print(train_df.shape, val_df.shape, test_df.shape)

(2412, 2) (426, 2) (715, 2)


# Authentication

A Hugging Face access token is securely retrieved using Kaggle Secrets.

This allows authenticated access to pretrained transformer models without exposing credentials inside the notebook.

In [9]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

# Loading the Pretrained MentalRoBERTa Model

The pretrained **MentalRoBERTa** model is loaded from Hugging Face.

Configuration:
- Backbone: MentalRoBERTa
- Classification Task: Binary Stress Detection
- Number of Output Labels: 2

A tokenizer is also initialized to convert raw text into token IDs that can be processed by the transformer model.

Maximum sequence length is fixed at 256 tokens.

In [10]:
MODEL_NAME = "mental/mental-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: mental/mental-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Dataset Tokenization

The Pandas DataFrames are converted into Hugging Face Dataset objects.

Each sample is tokenized using the MentalRoBERTa tokenizer.

Tokenization includes:

- Padding
- Truncation
- Maximum sequence length of 256 tokens

The resulting datasets are formatted as PyTorch tensors for efficient GPU training.

In [11]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df.rename(columns={"label":"labels"}).reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df.rename(columns={"label":"labels"}).reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.rename(columns={"label":"labels"}).reset_index(drop=True))

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/2412 [00:00<?, ? examples/s]

Map:   0%|          | 0/426 [00:00<?, ? examples/s]

Map:   0%|          | 0/715 [00:00<?, ? examples/s]

# Fine-Tuning on Dreaddit

The MentalRoBERTa backbone is fine-tuned on the Dreaddit dataset.

Training configuration:

- Optimizer: AdamW
- Learning Rate: 2 × 10⁻⁵
- Batch Size: 16
- Number of Epochs: 3
- Evaluation Strategy: Every Epoch

Macro F1-score is selected as the primary evaluation metric because it is more suitable than accuracy for potentially imbalanced mental health datasets.

Training and validation losses are monitored after every epoch.

In [12]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {"f1_macro": f1_score(labels, preds, average="macro")}

args = TrainingArguments(
    output_dir="/kaggle/working/dreaddit_ckpt",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1 Macro
1,No log,0.810004,0.820258
2,No log,0.791177,0.827474
3,No log,0.810615,0.837468
4,No log,1.031931,0.822747


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=304, training_loss=0.640300851119192, metrics={'train_runtime': 164.0885, 'train_samples_per_second': 58.798, 'train_steps_per_second': 1.853, 'total_flos': 634623865528320.0, 'train_loss': 0.640300851119192, 'epoch': 4.0})

In [13]:
test_results = trainer.predict(test_ds)
test_preds = test_results.predictions.argmax(-1)
print("Test F1-macro:", f1_score(test_df["label"], test_preds, average="macro"))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Test F1-macro: 0.8192183080785808


# Model Evaluation

After training, the model is evaluated on the held-out test dataset.

Performance Metric:
- Macro F1-score

Result:

**Test Macro F1-score = 0.819**

This indicates that the fine-tuned MentalRoBERTa model successfully learned stress-related linguistic patterns from the Dreaddit dataset and serves as a strong initialization for the next stage of Phase 1.

# Saving the Fine-Tuned Backbone

The trained MentalRoBERTa model and tokenizer are saved to the Kaggle working directory.

This checkpoint will be reused as the initialization point for the GoEmotions fine-tuning stage, enabling transfer learning instead of training from scratch.

In [14]:
model.save_pretrained("/kaggle/working/mentalroberta_dreaddit")
tokenizer.save_pretrained("/kaggle/working/mentalroberta_dreaddit")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/mentalroberta_dreaddit/tokenizer_config.json',
 '/kaggle/working/mentalroberta_dreaddit/tokenizer.json')

# Checkpoint Verification

A verification step confirms that the trained model checkpoint has been successfully saved.

The returned value `True` indicates that the checkpoint exists and is ready for reuse in subsequent training stages.

In [17]:
import os
print(os.path.exists("/kaggle/working/mentalroberta_dreaddit"))

True


# Loading the GoEmotions Dataset

The GoEmotions dataset is loaded from multiple CSV files and merged into a single dataset.

This dataset contains Reddit comments annotated with multiple emotion labels.

The combined dataset forms the basis for multi-label emotion classification.

In [18]:
import glob, pandas as pd

emotion_files = sorted(glob.glob("/kaggle/input/**/goemotions_*.csv", recursive=True))
print(emotion_files)

go_df = pd.concat([pd.read_csv(f) for f in emotion_files], ignore_index=True)
print(go_df.shape)
print(go_df.columns.tolist())

['/kaggle/input/datasets/shashwatkashyap12221/go-emotions/goemotions_1.csv', '/kaggle/input/datasets/shashwatkashyap12221/go-emotions/goemotions_2.csv', '/kaggle/input/datasets/shashwatkashyap12221/go-emotions/goemotions_3.csv']
(211225, 37)
['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id', 'created_utc', 'rater_id', 'example_very_unclear', 'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


# Emotion Label Processing

The original GoEmotions dataset contains multiple binary emotion columns.

The notebook:

- Selects the required emotion labels.
- Removes samples without emotion annotations.
- Prepares the dataset for multi-label classification.

Each training sample may contain more than one emotion simultaneously.

In [19]:
EMOTION_COLS = ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
                'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
                'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
                'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
                'relief', 'remorse', 'sadness', 'surprise', 'neutral']

# drop rows the annotators flagged as too unclear to label
go_clean = go_df[go_df["example_very_unclear"] == False].copy()

# raw file has one row per annotator rating of the same comment id; aggregate by id
agg = go_clean.groupby("id").agg({**{"text": "first"}, **{c: "max" for c in EMOTION_COLS}}).reset_index()

print(agg.shape)
agg.head()

(58009, 30)


,id,text,admiration,amusement,anger,annoyance,approval,caring,confusion,curiosity,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,eczazk6,Fast as [NAME] will carry me. Seriously uptown...,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,eczb07q,You blew it. They played you like a fiddle.,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,eczb4bm,TL;DR No more Superbowls for [NAME]. Get ready...,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,eczb527,So much time saved. Not.,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,eczb6r7,Emotes have a ridiculous amount of effort put ...,0,0,0,0,1,0,0,0,...,0,0,0,0,1,0,0,1,0,1


# Dataset Splitting

The processed GoEmotions dataset is divided into:

- Training Set
- Validation Set
- Test Set

This ensures unbiased evaluation while allowing model selection based on validation performance.

In [20]:
from sklearn.model_selection import train_test_split

train_emo, temp_emo = train_test_split(agg, test_size=0.15, random_state=42)
val_emo, test_emo = train_test_split(temp_emo, test_size=0.5, random_state=42)

print(train_emo.shape, val_emo.shape, test_emo.shape)

(49307, 30) (4351, 30) (4351, 30)


# Tokenization for Multi-Label Classification

The GoEmotions dataset is converted into Hugging Face Dataset format.

Each text sample is tokenized using the same MentalRoBERTa tokenizer.

Emotion labels are stored as multi-hot vectors, enabling simultaneous prediction of multiple emotions.

In [21]:
import numpy as np
from datasets import Dataset

def make_dataset(df):
    labels = df[EMOTION_COLS].values.astype(np.float32)
    ds = Dataset.from_dict({"text": df["text"].tolist(), "labels": labels.tolist()})
    return ds.map(tokenize, batched=True)

train_emo_ds = make_dataset(train_emo)
val_emo_ds = make_dataset(val_emo)
test_emo_ds = make_dataset(test_emo)

Map:   0%|          | 0/49307 [00:00<?, ? examples/s]

Map:   0%|          | 0/4351 [00:00<?, ? examples/s]

Map:   0%|          | 0/4351 [00:00<?, ? examples/s]

# Transfer Learning from Dreaddit

Instead of initializing from the original pretrained MentalRoBERTa model, the backbone learned from the Dreaddit stress classification task is reused.

Only the classification head is replaced to support multi-label emotion prediction.

This follows the Phase 1 training strategy of gradually adapting the transformer to mental health language before learning fine-grained emotions.

In [22]:
from transformers import AutoModelForSequenceClassification

NUM_EMOTIONS = len(EMOTION_COLS)

emo_model = AutoModelForSequenceClassification.from_pretrained(
    "/kaggle/working/mentalroberta_dreaddit",
    num_labels=NUM_EMOTIONS,
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True
).to(device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: /kaggle/working/mentalroberta_dreaddit
Key                        | Status   |                                                                                      
---------------------------+----------+--------------------------------------------------------------------------------------
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2, 768]) vs model:torch.Size([28, 768])
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2]) vs model:torch.Size([28])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


# Evaluation Metric

For multi-label emotion classification, predictions are converted into probabilities using the sigmoid activation function.

A threshold of 0.3 is used to determine active emotion labels.

Macro F1-score is calculated across all emotion categories to evaluate overall performance.

In [23]:
from sklearn.metrics import f1_score as f1_score_fn

def compute_metrics_multilabel(pred):
    labels = pred.label_ids
    logits = pred.predictions
    probs = 1 / (1 + np.exp(-logits))   # sigmoid
    preds = (probs >= 0.3).astype(int)  # threshold; GoEmotions is sparse so 0.3 works better than 0.5
    return {"f1_macro": f1_score_fn(labels, preds, average="macro", zero_division=0)}

# Fine-Tuning on GoEmotions

The MentalRoBERTa backbone is further fine-tuned on the GoEmotions dataset.

Training Configuration:

- Learning Rate: 2 × 10⁻⁵
- Batch Size: 16
- Epochs: 3
- Multi-label Binary Cross Entropy Loss

Validation performance is measured after every epoch to monitor learning progress.

In [24]:
emo_args = TrainingArguments(
    output_dir="/kaggle/working/goemotions_ckpt",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
)

emo_trainer = Trainer(
    model=emo_model,
    args=emo_args,
    train_dataset=train_emo_ds,
    eval_dataset=val_emo_ds,
    compute_metrics=compute_metrics_multilabel,
)
emo_trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1 Macro
1,0.424552,0.408772,0.440563
2,0.391773,0.392915,0.466694
3,0.378941,0.390039,0.479635


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=4623, training_loss=0.41859529937935575, metrics={'train_runtime': 2179.8634, 'train_samples_per_second': 67.858, 'train_steps_per_second': 2.121, 'total_flos': 9732183989916672.0, 'train_loss': 0.41859529937935575, 'epoch': 3.0})

# Validation Performance

Validation Macro F1-score is recorded after every epoch.

The increasing validation scores indicate that the model continues learning meaningful emotional representations while avoiding severe overfitting.

In [25]:
for log in emo_trainer.state.log_history:
    if "eval_f1_macro" in log:
        print(log)

{'eval_loss': 0.4087717831134796, 'eval_f1_macro': 0.4405625439958302, 'eval_runtime': 17.8969, 'eval_samples_per_second': 243.114, 'eval_steps_per_second': 3.8, 'epoch': 1.0, 'step': 1541}
{'eval_loss': 0.3929150700569153, 'eval_f1_macro': 0.4666936724658423, 'eval_runtime': 17.8141, 'eval_samples_per_second': 244.245, 'eval_steps_per_second': 3.817, 'epoch': 2.0, 'step': 3082}
{'eval_loss': 0.3900391459465027, 'eval_f1_macro': 0.4796346253878821, 'eval_runtime': 17.785, 'eval_samples_per_second': 244.644, 'eval_steps_per_second': 3.823, 'epoch': 3.0, 'step': 4623}


In [26]:
test_results_emo = emo_trainer.predict(test_emo_ds)
probs = 1 / (1 + np.exp(-test_results_emo.predictions))
preds = (probs >= 0.3).astype(int)
test_f1 = f1_score_fn(test_results_emo.label_ids, preds, average="macro", zero_division=0)
print("GoEmotions Test F1-macro:", test_f1)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


GoEmotions Test F1-macro: 0.48758117603436185


# Final Evaluation

The final model is evaluated on the unseen GoEmotions test dataset.

Result:

**Test Macro F1-score = 0.488**

Considering the difficulty of large-scale multi-label emotion classification involving numerous emotion categories, this performance demonstrates that the model successfully learned generalized emotional representations suitable for later phases of the Mental Health Monitoring System.

# Final Phase 1 Checkpoint

The fully fine-tuned MentalRoBERTa model is saved as the final output of Phase 1.

This checkpoint combines:

- Mental health language understanding from Dreaddit
- Fine-grained emotion recognition from GoEmotions

The saved model serves as the pretrained backbone for Phase 2, where Graph Attention Networks (GATs) will be introduced while keeping the transformer backbone frozen during graph training.

In [27]:
emo_model.save_pretrained("/kaggle/working/mentalroberta_phase1_final")
tokenizer.save_pretrained("/kaggle/working/mentalroberta_phase1_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/mentalroberta_phase1_final/tokenizer_config.json',
 '/kaggle/working/mentalroberta_phase1_final/tokenizer.json')